# 6. Balanced assignment with named students and projects

Lecture: *Week 6 - Integer Linear Programming - Further Problems*.

*Original script: `5 Source Files/Python/balanced_assignment_named.py`. Runs in the browser; Pyomo models use HiGHS instead of GLPK with the same answers.*

In [ ]:
# Run this cell first.
# In the browser (JupyterLite) it installs the extra packages into this tab;
# in Colab or Anaconda it installs them with pip (already installed = no change).
%pip install pyomo highspy

import sys
if sys.platform == "emscripten":  # only in the browser (JupyterLite): adapt Pyomo to it
    import contextlib, pyomo.common.tee, pyomo.common.formatting as _fmt
    # 1) browsers have no threads: skip Pyomo's threaded capture of solver output
    pyomo.common.tee.capture_output = lambda *args, **kwargs: contextlib.nullcontext()
    for _m in ("pyomo.contrib.appsi.solvers.highs", "pyomo.contrib.solver.solvers.highs"):
        if _m in sys.modules:
            sys.modules[_m].capture_output = pyomo.common.tee.capture_output
    # 2) the notebook's output returns None from write(); model.pprint()/display() need a number
    if not getattr(_fmt.StreamIndenter, "_browser_patched", False):
        class _Counted:
            def __init__(self, s): self._s = s
            def write(self, data): self._s.write(data); return len(data)
            def __getattr__(self, name): return getattr(self._s, name)
        def _init(self, ostream, indent=" " * 4, _orig=_fmt.StreamIndenter.__init__):
            _orig(self, ostream, indent)
            if not isinstance(self.target_os, _Counted):
                object.__setattr__(self, "target_os", _Counted(self.target_os))
        _fmt.StreamIndenter.__init__ = _init
        _fmt.StreamIndenter._browser_patched = True

In [ ]:
from pyomo.environ import *

In [ ]:
# Define the binary variables
model = ConcreteModel()
#2 range arguments to give coordinates range(0)is collumn range (1) is row
workers = range(4)
jobs = range(4)
model.x = Var(workers, jobs, within=Binary)

time = [
    [9, 2, 7, 8],
    [6, 4, 3, 7],
    [5, 8, 1, 8],
    [7, 6, 9, 4]
]

In [ ]:
# Create the optimization problem
#minimise the number cosen in regards to the weight
model.obj = Objective(expr=sum(sum(time[worker][job] * model.x[worker, job] for job in jobs) for worker in workers), sense=minimize)

In [ ]:
# Define the equality constraints for workers and jobs
model.worker_constraints = ConstraintList()
model.job_constraints = ConstraintList()

In [ ]:
#creates binary variables procedurally
for worker in workers:      # for each worker
    model.worker_constraints.add(sum(model.x[worker, job] for job in jobs) == 1)     # only assign a single job to a worker
    
for job in jobs:	    # for each job
    model.job_constraints.add(sum(model.x[worker, job] for worker in workers) == 1)  # only assign a single worker to a job

In [ ]:
# Create a solver
solver = SolverFactory('appsi_highs')

In [ ]:
# Display the model
model.pprint()

In [ ]:
# Solve the model
solver.solve(model, tee=False)  # Set tee=True to display solver output

In [ ]:
# Display the results
model.display()